# **AI ASSIGNMENT # 02**
## **I242533**
## **Dawar Ahmed** 
## **DS - *B***

In [1]:
import random

#Card Class
class Card:
    def __init__(self, color, value):
        self.color = color
        self.value = str(value)

    def __repr__(self):
        return f"{self.color} {self.value}"
    
    #Check if two cards match for playing
    def matches(self, other_card):
        return self.color == other_card.color or self.value == other_card.value

#Deck Generator
def generate_deck():
    colors = ['Red', 'Blue', 'Green', 'Yellow']
    numbers = list(range(10))
    
    deck = []
    for color in colors:
        for num in numbers:
            deck.append(Card(color, num))
        #Adding Skip cards
        deck.append(Card(color, 'Skip'))
        
    random.shuffle(deck)
    return deck

#Legal Move Generator
def get_valid_moves(hand, top_card):
    valid_moves = []
    for card in hand:
        if card.matches(top_card):
            valid_moves.append(card)
    return valid_moves

#Testing Step 1
deck = generate_deck()
player1_hand = [deck.pop() for _ in range(5)]
top_card = Card('Red', '5')

print("Top Card on Table:", top_card)
print("Player 1 Hand:", player1_hand)
print("Valid Moves for Player 1:", get_valid_moves(player1_hand, top_card))

Top Card on Table: Red 5
Player 1 Hand: [Red 9, Red 3, Red 2, Blue Skip, Yellow 0]
Valid Moves for Player 1: [Red 9, Red 3, Red 2]


In [2]:
#GameState class
class GameState:
    def __init__(self, ai_cards, opponent1_card_count, opponent2_card_count, top_card, deck_size):
        self.ai_cards = ai_cards
        self.opponent1_cards = opponent1_card_count
        self.opponent2_cards = opponent2_card_count
        self.top_card = top_card
        self.deck = deck_size
        
    def __repr__(self):
        return f"Top: {self.top_card} | AI Cards: {len(self.ai_cards)} | Opp1: {self.opponent1_cards} | Opp2: {self.opponent2_cards}"

#Evaluation Function
def evaluate_state(state, strategy="Defensive"):
    #Formula variables
    c_ai = len(state.ai_cards)
    c_opp = (state.opponent1_cards + state.opponent2_cards) / 2.0
    s_count = sum(1 for card in state.ai_cards if card.value == 'Skip')
    
    #Base Score
    score = 50 - (5 * c_ai) + (2 * c_opp) + (3 * s_count)
    
    #Strategy Tuning
    if strategy == "Defensive":
        if state.opponent1_cards <= 1 or state.opponent2_cards <= 1:
            score -= 20
        score += (2 * s_count)
        
    elif strategy == "Offensive":
        if c_ai <= 1:
            score += 20
        score -= (2 * c_ai)
        
    return score

#Testing Step 2
fake_ai_hand = [Card('Red', '5'), Card('Blue', 'Skip'), Card('Green', '9')]
fake_state = GameState(fake_ai_hand, 2, 4, Card('Red', '3'), 30)

print(fake_state)
print("Defensive AI Score:", evaluate_state(fake_state, "Defensive"))
print("Offensive AI Score:", evaluate_state(fake_state, "Offensive"))

Top: Red 3 | AI Cards: 3 | Opp1: 2 | Opp2: 4
Defensive AI Score: 46.0
Offensive AI Score: 38.0


In [20]:
#Simulate Move Function
def simulate_move(state, move, player_turn):
    new_ai_cards = state.ai_cards.copy()
    new_opponent1_cards = state.opponent1_cards
    new_opponent2_cards = state.opponent2_cards
    new_top_card = state.top_card
    new_deck_size = state.deck
    
    #AI Turn
    if player_turn == 0:
        if move == "Draw":
            if new_deck_size > 0:
                new_ai_cards.append(Card('Unknown', 'Unknown')) 
                new_deck_size -= 1
        else:
            for c in new_ai_cards:
                if c.color == move.color and c.value == move.value:
                    new_ai_cards.remove(c)
                    new_top_card = move
                    break
                    
    #Opponent 1 Turn
    elif player_turn == 1:
        if move == "Draw":
            if new_deck_size > 0:
                new_opponent1_cards += 1
                new_deck_size -= 1
        else:
            new_opponent1_cards -= 1
            new_top_card = move
            
    #Opponent 2 Turn
    elif player_turn == 2:
        if move == "Draw":
            if new_deck_size > 0:
                new_opponent2_cards += 1
                new_deck_size -= 1
        else:
            new_opponent2_cards -= 1
            new_top_card = move
            
    return GameState(new_ai_cards, new_opponent1_cards, new_opponent2_cards, new_top_card, new_deck_size)

#Testing Step 3
test_state = GameState([Card('Red', '5'), Card('Blue', 'Skip')], 3, 2, Card('Red', '3'), 25)
print("Test 1 AI plays card:", simulate_move(test_state, Card('Red', '5'), 0))
print("Test 2 AI draws card:", simulate_move(test_state, "Draw", 0))



Test 1 AI plays card: Top: Red 5 | AI Cards: 1 | Opp1: 3 | Opp2: 2
Test 2 AI draws card: Top: Red 3 | AI Cards: 3 | Opp1: 3 | Opp2: 2


In [21]:
#Minimax Algorithm (Defensive AI)
def minimax(state, depth, alpha, beta, player_turn):
    #Base case
    if depth == 0 or len(state.ai_cards) == 0 or state.opponent1_cards == 0 or state.opponent2_cards == 0:
        return evaluate_state(state, "Defensive"), None
        
    #AI Turn (Maximizing Player)
    if player_turn == 0:
        max_eval = float('-inf')
        best_move = None
        
        valid_moves = get_valid_moves(state.ai_cards, state.top_card)
        if not valid_moves:
            valid_moves = ["Draw"]
            
        for move in valid_moves:
            new_state = simulate_move(state, move, player_turn)
            eval_score, _ = minimax(new_state, depth - 1, alpha, beta, 1)
            
            if eval_score > max_eval:
                max_eval = eval_score
                best_move = move
                
            alpha = max(alpha, eval_score)
            if beta <= alpha:
                break 
        return max_eval, best_move
        
    #Opponents Turn (Minimizing Players)
    else:
        min_eval = float('inf')
        possible_moves = [state.top_card, "Draw"]
        
        for move in possible_moves:
            new_state = simulate_move(state, move, player_turn)
            next_turn = 2 if player_turn == 1 else 0
            
            eval_score, _ = minimax(new_state, depth - 1, alpha, beta, next_turn)
            
            if eval_score < min_eval:
                min_eval = eval_score
                
            beta = min(beta, eval_score)
            if beta <= alpha:
                break 
        return min_eval, None

#Testing Step 4
best_score, best_move = minimax(test_state, 3, float('-inf'), float('inf'), 0)
print("Minimax Best Move:", best_move)
print("Expected Score:", best_score)

Minimax Best Move: Red 5
Expected Score: 33.0


In [22]:
#Expectimax Algorithm (Offensive AI)
def expectimax(state, depth, player_turn):
    #Base case
    if depth == 0 or len(state.ai_cards) == 0 or state.opponent1_cards == 0 or state.opponent2_cards == 0:
        return evaluate_state(state, "Offensive"), None
        
    #AI Turn (Maximizing Player)
    if player_turn == 0:
        max_eval = float('-inf')
        best_move = None
        
        valid_moves = get_valid_moves(state.ai_cards, state.top_card)
        if not valid_moves:
            valid_moves = ["Draw"]
            
        for move in valid_moves:
            new_state = simulate_move(state, move, player_turn)
            eval_score, _ = expectimax(new_state, depth - 1, 1)
            
            if eval_score > max_eval:
                max_eval = eval_score
                best_move = move
                
        return max_eval, best_move
        
    #Chance Node (Opponents)
    else:
        expected_eval = 0
        possible_moves = [state.top_card, "Draw"]
        
        for move in possible_moves:
            new_state = simulate_move(state, move, player_turn)
            next_turn = 2 if player_turn == 1 else 0
            
            eval_score, _ = expectimax(new_state, depth - 1, next_turn)
            expected_eval += eval_score
            
        expected_eval = expected_eval / len(possible_moves)
        return expected_eval, None

#Testing Step 5
best_exp_score, best_exp_move = expectimax(test_state, 3, 0)
print("Expectimax Best Move:", best_exp_move)
print("Expectimax Expected Score:", best_exp_score)

Expectimax Best Move: Red 5
Expectimax Expected Score: 71.0


In [25]:
#Final Game Loop Simulation
def play_uno():
    print("Starting UNO AI Game")
    deck = generate_deck()
    
    #Distribute 5 cards to each player
    p1_hand = [deck.pop() for _ in range(5)]
    p2_hand = [deck.pop() for _ in range(5)]
    p3_hand = [deck.pop() for _ in range(5)]
    
    top_card = deck.pop()
    current_player = 0
    
    while True:
        #Check win conditions
        if len(p1_hand) == 0:
            print("Player 1 Wins")
            break
        elif len(p2_hand) == 0:
            print("Player 2 Wins")
            break
        elif len(p3_hand) == 0:
            print("Player 3 Wins")
            break
        if len(deck) == 0:
            print("Draw Deck Empty")
            break
            
        print("Top card:", top_card)
        
        #Player 1 Turn (Defensive AI)
        if current_player == 0:
            print("Player 1 Defensive AI Turn")
            print("P1 hand:", p1_hand)
            state = GameState(p1_hand, len(p2_hand), len(p3_hand), top_card, len(deck))
            score, move = minimax(state, 3, float('-inf'), float('inf'), 0)
            
            print("P1 decisions considered at depth 1:")
            valid_moves = get_valid_moves(p1_hand, top_card)
            if not valid_moves:
                valid_moves = ["Draw"]
            for v_move in valid_moves:
                test_state = simulate_move(state, v_move, 0)
                eval_score, _ = minimax(test_state, 2, float('-inf'), float('inf'), 1)
                print(f"Play: {v_move} Expected score: {eval_score}")
                
            if move == "Draw" or move is None:
                print("Action: P1 Draws a card")
                p1_hand.append(deck.pop())
                current_player = 1
            else:
                print("Action: P1 Plays", move)
                for c in p1_hand:
                    if c.color == move.color and c.value == move.value:
                        p1_hand.remove(c)
                        break
                top_card = move
                if top_card.value == 'Skip':
                    print("Player 2 Skipped")
                    current_player = 2
                else:
                    current_player = 1
                    
        #Player 2 Turn (Offensive AI)
        elif current_player == 1:
            print("Player 2 Offensive AI Turn")
            print("P2 hand:", p2_hand)
            state = GameState(p2_hand, len(p3_hand), len(p1_hand), top_card, len(deck))
            score, move = expectimax(state, 3, 0)
            
            print("P2 decisions considered at depth 1:")
            valid_moves = get_valid_moves(p2_hand, top_card)
            if not valid_moves:
                valid_moves = ["Draw"]
            for v_move in valid_moves:
                test_state = simulate_move(state, v_move, 0)
                eval_score, _ = expectimax(test_state, 2, 1)
                print(f"Play: {v_move} Expected score: {eval_score}")
                
            if move == "Draw" or move is None:
                print("Action: P2 Draws a card")
                p2_hand.append(deck.pop())
                current_player = 2
            else:
                print("Action: P2 Plays", move)
                for c in p2_hand:
                    if c.color == move.color and c.value == move.value:
                        p2_hand.remove(c)
                        break
                top_card = move
                if top_card.value == 'Skip':
                    print("Player 3 Skipped")
                    current_player = 0
                else:
                    current_player = 2
                    
        #Player 3 Turn (Simulation Mode)
        elif current_player == 2:
            print("Player 3 Simulation AI Turn")
            print("P3 hand:", p3_hand)
            state = GameState(p3_hand, len(p1_hand), len(p2_hand), top_card, len(deck))
            score, move = minimax(state, 3, float('-inf'), float('inf'), 0)
            
            if move == "Draw" or move is None:
                print("Action: P3 Draws a card")
                p3_hand.append(deck.pop())
                current_player = 0
            else:
                print("Action: P3 Plays", move)
                for c in p3_hand:
                    if c.color == move.color and c.value == move.value:
                        p3_hand.remove(c)
                        break
                top_card = move
                if top_card.value == 'Skip':
                    print("Player 1 Skipped")
                    current_player = 1
                else:
                    current_player = 0

#Run the Game
play_uno()

Starting UNO AI Game
Top card: Yellow 1
Player 1 Defensive AI Turn
P1 hand: [Blue Skip, Blue 1, Red 9, Yellow Skip, Blue 6]
P1 decisions considered at depth 1:
Play: Blue 1 Expected score: 48.0
Play: Yellow Skip Expected score: 43.0
Action: P1 Plays Blue 1
Top card: Blue 1
Player 2 Offensive AI Turn
P2 hand: [Green 2, Blue 7, Yellow 2, Yellow 6, Red 2]
P2 decisions considered at depth 1:
Play: Blue 7 Expected score: 31.0
Action: P2 Plays Blue 7
Top card: Blue 7
Player 3 Simulation AI Turn
P3 hand: [Blue 2, Red 6, Blue 8, Green 0, Red 4]
Action: P3 Plays Blue 2
Top card: Blue 2
Player 1 Defensive AI Turn
P1 hand: [Blue Skip, Red 9, Yellow Skip, Blue 6]
P1 decisions considered at depth 1:
Play: Blue Skip Expected score: 46.0
Play: Blue 6 Expected score: 51.0
Action: P1 Plays Blue 6
Top card: Blue 6
Player 2 Offensive AI Turn
P2 hand: [Green 2, Yellow 2, Yellow 6, Red 2]
P2 decisions considered at depth 1:
Play: Yellow 6 Expected score: 36.0
Action: P2 Plays Yellow 6
Top card: Yellow 6
Pl

### Conclusion: Minimax vs. Expectimax

Based on the UNO game simulation, **Expectimax** generally performs better and acts as a more realistic AI compared to Minimax.

**Why?**
1. **Nature of UNO:** UNO is a stochastic game (game of chance) with imperfect information. We do not know the hidden cards of our opponents, nor do we know which card will be drawn from the deck next.
2. **Minimax Limitation:** Minimax assumes the opponent is playing optimally to minimize our score. However, in UNO, opponents cannot play "perfectly" against us because they don't know our cards. Minimax prepares for the absolute worst-case scenario, which makes it overly defensive and often leads to sub-optimal, slow gameplay.
3. **Expectimax Advantage:** Expectimax uses chance nodes to model the opponents' unknown moves as probabilities. It calculates the *average* expected outcome rather than assuming the worst-case. This aligns perfectly with UNO, where players make moves based on their own hands rather than purely trying to block one specific person. Expectimax takes calculated risks and sheds cards much more efficiently.